# Problem Setup
Here we present the usage of pytorch's automatic differentiation algorithm.

We use the model trained in the previous section in order to obtain its derivatives through the automatic differentiation algorithm:

$$f(x)=sin(x)$$

$$\frac{d f(x)}{d x}=cos(x)$$

# Libraries

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.autograd import grad

from Lib.ImportLibrary import configure_notebook
from Lib.MultiLayerNet import MultiLayerNet

dev = configure_notebook()
output_dir = Path("output/Case1")
checkpoint = output_dir / "model.pt"

# Training parameters

In [ ]:
N = 500

# Functions

In [ ]:
def f(x):
    y=np.sin(x)
    return y 

def df(x):
    dy = np.cos(x)
    return dy

In [ ]:
def mygrad(y, x):
    dydx = grad(y, x, torch.ones_like(y),
                create_graph=True, retain_graph=True)[0]
    return dydx

# Load Model

In [ ]:
if not checkpoint.is_file():
    checkpoint = Path("checkpoints/Case1.pt")
if not checkpoint.is_file():
    raise FileNotFoundError("Run Notebook 1 to create output/Case1/model.pt.")

# Set Neural Network
layers = [1,30,30,30,1]
model = MultiLayerNet(layers, init=True)
model.to(dev)
start_time = time.time()

model.load_state_dict(torch.load(checkpoint, map_location=dev, weights_only=True))
model.eval()

In [ ]:
x_test = torch.linspace(0, 2 * np.pi, N, device=dev).view(-1,1)
x_test.requires_grad_(True)

y_pred = model(x_test)
dy_pred = mygrad(y_pred, x_test)

x_test = x_test.detach().cpu().numpy()
y_real = f(x_test)
dy_real = df(x_test)

y_pred = y_pred.detach().cpu().numpy()
dy_pred = dy_pred.detach().cpu().numpy()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(20, 8))
ax[0].plot(x_test,y_real,color='blue',label='Real')
ax[0].plot(x_test,y_pred,':',color='red',label='Pred')
ax[0].set_xlabel('x',color='black')
ax[0].set_ylabel('f(x)',color='black')
ax[0].tick_params(axis='y', color='black')
ax[0].legend(loc = 'upper left')

ax[1].plot(x_test,dy_real,color='blue',label='Real')
ax[1].plot(x_test,dy_pred,':',color='red',label='Pred')
ax[1].set_xlabel('x',color='black')
ax[1].set_ylabel('df(x)',color='black')
ax[1].tick_params(axis='y', color='black')
ax[1].legend(loc = 'upper left')

In [ ]:
print('MSE for f(x):', np.mean(np.square(y_real-y_pred)))
print('MSE for df(x):', np.mean(np.square(dy_real-dy_pred)))